In [7]:
# import geemap, ee
import ee

try:
        ee.Initialize()
except Exception as e:
        ee.Authenticate()
        ee.Initialize()

In [8]:
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt 
import geopandas as gpd
# import folium
import json
import os
import csv
from datetime import datetime, timedelta

In [ ]:
tiles_25 = gpd.read_file(TILES_SHP)

In [9]:
tiles_25 = gpd.read_file('/data/afg_satellite/Grids/AFG_025_grid.shp')

In [10]:
def shp_to_fc(file)->ee.FeatureCollection:
    """
    convert simple polygon/ shapefile to earth engine feature collection
    Note: doesn't seem to work very well with disjoint polygons/ nested polygons
    """
    g = [i for i in file.geometry]
    features=[]
    for i in range(len(g)):
        g = [i for i in file.geometry]
        x,y = g[i].exterior.coords.xy
        cords = np.dstack((x,y)).tolist()

        g=ee.Geometry.Polygon(cords)
        feature = ee.Feature(g)
        features.append(feature)
#     print("done")

    ee_object = ee.FeatureCollection(features)
    return ee_object

In [21]:
# Does all the work needed, taking an input of a specific tile number, and the year to request data for,
# and returning an eeList of each pixel and their peak NDVI timestep.
# Takes about 8 seconds to run for a given tile and year

def imagetobandsAg(tile, year, applyAgMask = True):
    top = tiles_25.iloc[tile]["top"]
    bottom = tiles_25.iloc[tile]["bottom"]
    left = tiles_25.iloc[tile]["left"]
    right = tiles_25.iloc[tile]["right"]
    tilegeo = shp_to_fc(tiles_25[tiles_25.index==tile]).geometry()

    mod_dataset=ee.ImageCollection('MODIS/006/MOD13Q1').filter(ee.Filter.date('2018-01-01', '2018-12-31')) 
    # just for set-up, doesn't matter which year
    mod_dataset_NDVI = mod_dataset.select("NDVI")
    mod_image_NDVI = mod_dataset_NDVI.first()
    mod_image_NDVI_U = mod_image_NDVI.unmask(-9999)
    
    # this part does the actual work
    if (applyAgMask == True):
        if (year == 2014):
            landCover = ee.Image("COPERNICUS/Landcover/100m/Proba-V-C3/Global/2015")
        elif (year == 2020):
            landCover = ee.Image("COPERNICUS/Landcover/100m/Proba-V-C3/Global/2019")
        else:
            landCover = ee.Image("COPERNICUS/Landcover/100m/Proba-V-C3/Global/" + str(year))
        agMask = landCover.select('discrete_classification')\
            .updateMask(landCover.select(['discrete_classification']).neq(40))\
            .mask()\
            .Not() 
        mod_image_NDVI_U = mod_image_NDVI_U.updateMask(agMask) # here mask according to land cover  
    clipped = mod_image_NDVI_U.clip(tilegeo).multiply(0.0001) # .0001 is the scale for NDVI 
    #(https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13Q1#bands)

    # note: sample function drops masked pixels, so non-agricultural pixels are dropped
    # (see https://developers.google.com/earth-engine/apidocs/ee-image-sample)
    pixsample = clipped.sample(region = tilegeo,
                               scale = 250, 
                               projection = 'SR-ORG:6974', 
                               geometries = True, 
                               factor = 1)
    pixsampsize = pixsample.size().getInfo()
    pixellist = pixsample.toList(pixsampsize)
    
    return pixellist, pixsampsize

In [12]:
# Functions to extract information from the images' getInfo dictionaries

def getlong(dicti):
    return dicti.get('coordinates')[0]

def getlat(dicti):
    return dicti.get('coordinates')[1]


In [ ]:
# Gathers data for the specified tiles and years. Runtime is about 1 minute per tile. Saves data to the specified
# filepath at the bottom under "filename". Data for tile XXX will be saved in that folder as XXXdata.csv

for tile in alltiles:
    count = 0

    for year in allyears:
        if (count == 0): # just for set up
            ps, nrow = imagetobandsAg(tile, year, applyAgMask = False)
            jsont = json.dumps(ps.getInfo())
            dft = pd.read_json(jsont)
            longs = dft['geometry'].apply(getlong)
            lats = dft['geometry'].apply(getlat)
            masterdf = pd.DataFrame({'Latitude': lats, 'Longitude': longs})

        ps, nrow = imagetobandsAg(tile, year) # this only has agricultural pixels according to copernicus
        if (nrow != 0):
            jsont = json.dumps(ps.getInfo())
            dft = pd.read_json(jsont)
            longs = dft['geometry'].apply(getlong)
            lats = dft['geometry'].apply(getlat)
            dff = pd.DataFrame({'Latitude': lats, 'Longitude': longs, "Agr_" + str(year): 1}) 
            # line above: sets Agr_year to be 1 for rows that appear in lats and longs, i.e., if data are non-missing
            masterdf = masterdf.merge(dff, how = "outer", on = ["Latitude", "Longitude"])

        count += 1
    
    filename = os.path.join(AGR_YN_DIR, str(tile) + "data.csv")
    masterdf.to_csv(filename, index = None)

In [22]:
# Gathers data for the specified tiles and years. Runtime is about 1 minute per tile. Saves data to the specified
# filepath at the bottom under "filename". Data for tile XXX will be saved in that folder as XXXdata.csv

for tile in alltiles:
    count = 0

    for year in allyears:
        if (count == 0): # just for set up
            ps, nrow = imagetobandsAg(tile, year, applyAgMask = False)
            jsont = json.dumps(ps.getInfo())
            dft = pd.read_json(jsont)
            longs = dft['geometry'].apply(getlong)
            lats = dft['geometry'].apply(getlat)
            masterdf = pd.DataFrame({'Latitude': lats, 'Longitude': longs})

        ps, nrow = imagetobandsAg(tile, year) # this only has agricultural pixels according to copernicus
        if (nrow != 0):
            jsont = json.dumps(ps.getInfo())
            dft = pd.read_json(jsont)
            longs = dft['geometry'].apply(getlong)
            lats = dft['geometry'].apply(getlat)
            dff = pd.DataFrame({'Latitude': lats, 'Longitude': longs, "Agr_" + str(year): 1}) 
            # line above: sets Agr_year to be 1 for rows that appear in lats and longs, i.e., if data are non-missing
            masterdf = masterdf.merge(dff, how = "outer", on = ["Latitude", "Longitude"])

        count += 1
    
    filename = "/data/afg_satellite/bestdates/agrYN/" + str(tile) + "data.csv"
    masterdf.to_csv(filename, index = None)

In [ ]:
# takes 1.5 hours